In [2]:
import pandas as pd
import numpy as np

# 1. Defina o nome do arquivo que você baixou do Banco Central
nome_arquivo_entrada = 'STP-20260425102741818.csv' # Mude para o nome do arquivo baixado

# 2. Ler o arquivo 
# Como o título do BCB vem com caracteres estranhos (cmbio), pulamos a linha 0 (skiprows=1) 
# e forçamos os nomes limpos das colunas.
df = pd.read_csv(nome_arquivo_entrada, 
                 sep=';', 
                 encoding='latin1', # ou 'utf-8' dependendo de como baixou
                 skiprows=1, 
                 names=['Data', 'Taxa_Cambio', 'Taxa_Selic'])

# 3. Limpeza dos Dados
# O BCB usa '-' para dias sem cotação (finais de semana/feriados). Trocamos por vazio (NaN).
df.replace('-', np.nan, inplace=True)

# Limpar espaços invisíveis e forçar o Python a entender a vírgula como ponto decimal
df['Taxa_Cambio'] = df['Taxa_Cambio'].astype(str).str.strip().str.replace(',', '.')
df['Taxa_Selic'] = df['Taxa_Selic'].astype(str).str.strip().str.replace(',', '.')

# Transformar de texto para número real (ignora os valores vazios automaticamente)
df['Taxa_Cambio'] = pd.to_numeric(df['Taxa_Cambio'], errors='coerce')
df['Taxa_Selic'] = pd.to_numeric(df['Taxa_Selic'], errors='coerce')

# 4. Tratamento de Tempo (Extraindo o Ano)
# 4. Tratamento de Tempo (Extraindo o Ano)
# errors='coerce' força qualquer lixo em texto (como "Fonte") a virar NaT (Not a Time)
df['Data'] = pd.to_datetime(df['Data'], format='%d/%m/%Y', errors='coerce')

# Removemos todas as linhas que viraram NaT (o rodapé inútil)
df = df.dropna(subset=['Data'])

# Agora sim podemos extrair o ano com segurança
df['Ano'] = df['Data'].dt.year
# 5. Matemática Financeira: Agrupar por Ano e tirar a Média Simples
df_anual = df.groupby('Ano').agg({
    'Taxa_Cambio': 'mean',
    'Taxa_Selic': 'mean'
}).reset_index()

# Arredondando para ficar legível (4 casas para câmbio, 2 para Selic)
df_anual['Taxa_Cambio'] = df_anual['Taxa_Cambio'].round(4)
df_anual['Taxa_Selic'] = df_anual['Taxa_Selic'].round(2)

print("--- Visualização dos Primeiros Anos ---")
display(df_anual.head())

# 6. Exportar para um novo CSV (Pronto para cruzar com USDA)
nome_arquivo_saida = 'juros_cambio.csv'
df_anual.to_csv(nome_arquivo_saida, index=False)

print(f"\n✅ Sucesso! Arquivo limpo e anualizado salvo como: '{nome_arquivo_saida}'")

--- Visualização dos Primeiros Anos ---


,Ano,Taxa_Cambio,Taxa_Selic
0,2000,1.8295,17.60
1,2001,2.3522,17.46
2,2002,2.9309,19.22
3,2003,3.0715,23.52
4,2004,2.9257,16.38



✅ Sucesso! Arquivo limpo e anualizado salvo como: 'juros_cambio.csv'


In [3]:

# 1. Nome do arquivo (ajuste conforme o seu arquivo)
nome_arquivo_ipca = 'STP-20260425102807064.csv'

# 2. Ler o arquivo (pulando a primeira linha de títulos estranhos)
df_agro = pd.read_csv(nome_arquivo_ipca, 
                      sep=';', 
                      encoding='latin1', 
                      skiprows=1, 
                      names=['Data', 'IPCA_Mensal', 'IC_Br_Agro'])

# 3. Limpeza dos Dados
df_agro.replace('-', np.nan, inplace=True)
df_agro['IPCA_Mensal'] = df_agro['IPCA_Mensal'].astype(str).str.strip().str.replace(',', '.')
df_agro['IC_Br_Agro'] = df_agro['IC_Br_Agro'].astype(str).str.strip().str.replace(',', '.')

df_agro['IPCA_Mensal'] = pd.to_numeric(df_agro['IPCA_Mensal'], errors='coerce')
df_agro['IC_Br_Agro'] = pd.to_numeric(df_agro['IC_Br_Agro'], errors='coerce')

# 4. Tratamento de Data (Formato MM/YYYY)
# errors='coerce' limpa o rodapé "Fonte" automaticamente
df_agro['Data'] = pd.to_datetime(df_agro['Data'], format='%m/%Y', errors='coerce')
df_agro = df_agro.dropna(subset=['Data'])
df_agro['Ano'] = df_agro['Data'].dt.year

# 5. Matemática Financeira e Agregação
# IPCA: Vamos transformar em fator (ex: 0.62 -> 1.0062) para acumular
df_agro['IPCA_Fator'] = (df_agro['IPCA_Mensal'] / 100) + 1

df_agro_anual = df_agro.groupby('Ano').agg({
    'IPCA_Fator': 'prod',        # Multiplica os fatores para acumular a inflação
    'IC_Br_Agro': 'mean'         # Média simples para o índice de commodities
}).reset_index()

# Volta o IPCA para formato de porcentagem anual: (Fator_Final - 1) * 100
df_agro_anual['IPCA_Anual_Pct'] = (df_agro_anual['IPCA_Fator'] - 1) * 100

# Limpeza final: remover coluna auxiliar e arredondar
df_agro_anual = df_agro_anual.drop(columns=['IPCA_Fator'])
df_agro_anual['IPCA_Anual_Pct'] = df_agro_anual['IPCA_Anual_Pct'].round(2)
df_agro_anual['IC_Br_Agro'] = df_agro_anual['IC_Br_Agro'].round(2)

print("--- Dados Agropecuários e Inflação Anualizados ---")
display(df_agro_anual.head())

# 6. Exportar
nome_saida_agro = 'agro_inflacao_anual.csv'
df_agro_anual.to_csv(nome_saida_agro, index=False)

print(f"\n✅ Sucesso! Arquivo salvo como: '{nome_saida_agro}'")

--- Dados Agropecuários e Inflação Anualizados ---


,Ano,IC_Br_Agro,IPCA_Anual_Pct
0,2000,59.93,5.97
1,2001,71.84,7.67
2,2002,85.90,12.53
3,2003,103.49,9.30
4,2004,106.92,7.60



✅ Sucesso! Arquivo salvo como: 'agro_inflacao_anual.csv'
